# SigAlg's `SigmaAlgebra` class

In [1]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `SigmaAlgebra` class in SigAlg is the fundamental class for representing $\sigma$-algebras on sample spaces. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/core/#sigalg.core.SigmaAlgebra).

## Mathematical definition

A *$\sigma$-algebra* $\mathcal{F}$ on a set $\Omega$ is a collection of subsets of $\Omega$ that contains $\Omega$, and is closed under complementation and countable unions. In the case that $\Omega$ is finite (as it always is, in SigAlg), then $\mathcal{F}$ obviously needs only to be closed under finite unions.

A $\sigma$-algebra $\mathcal{F}$ determines its *atoms*, which are the nonempty sets $A\in \mathcal{F}$ that are *minimal* with respect to subset inclusion, in the following sense: if $B\in \mathcal{F}$ is nonempty and $B\subset A$, then necessarily $A=B$. And conversely, provided that $\Omega$ is finite, the $\sigma$-algebra $\mathcal{F}$ is completely recoverable from its atoms, in the sense that every event $A\in \mathcal{F}$ is a disjoint union of atoms.

If $\{A_i\}_{i\in I}$ is the set of atoms, indexed by a finite set $I$, then there is a mapping $\Omega \to I$ given by $\omega \mapsto i$, where $A_i$ is the unique atom that contains $\omega$. This mapping is what SigAlg uses to represent $\sigma$-algebras. The indices in $I$ are called *atom identifiers*.

In SigAlg, an instance `F` of `SigmaAlgebra` represents such a $\sigma$-algebra $\mathcal{F}$.

## Creating $\sigma$-algebras

### From dictionaries

The primary way to create a $\sigma$-algebra in SigAlg is to use the `from_dict` method, passing a dictionary that represents the atom identifier mapping $\Omega \to I$ described above. For example, let's consider a sample space $\Omega = \{0, 1, 2, 3, 4\}$ and a $\sigma$-algebra $\mathcal{F}$ with atoms $A_0 = \{0, 1, 2\}$, $A_1 = \{3\}$, and $A_2 = \{4\}$. Notice that the set $I$ of atom identifiers is $\{0,1,2\}$. In SigAlg, we write:

In [2]:
from sigalg.core import SampleSpace, SigmaAlgebra

Omega = SampleSpace().from_sequence(size=5)
F = SigmaAlgebra(sample_space=Omega).from_dict(
    {
        0: 0,
        1: 0,
        2: 0,
        3: 1,
        4: 2,
    }
)

print(F)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             0
3             1
4             2


Notice the sample space $\Omega$ was explicitly provided to the constructor before calling `from_dict`. This is optional, however. If it is not provided, it will be automatically generated from the dictionary keys. For example:

In [3]:
G = SigmaAlgebra(name="G").from_dict(
    {
        0: 0,
        1: 0,
        2: 0,
        3: 1,
        4: 2,
    }
)
G.sample_space.name = "Omega_generated"

print(G, "\n")
print(f"Generated sample space: {G.sample_space}")

Sigma algebra 'G':
        atom ID
sample         
0             0
1             0
2             0
3             1
4             2 

Generated sample space: Sample space 'Omega_generated':
[0, 1, 2, 3, 4]


### From `pd.Series` objects

Alternatively, we can create a $\sigma$-algebra from a `pd.Series` object, where the index of the series is the sample space $\Omega$ and the values of the series are the atom identifiers. For example:

In [4]:
import pandas as pd

atom_ids = pd.Series([0, 0, 0, 1, 2], index=["a", "b", "c", "d", "e"])
H = SigmaAlgebra(name="H").from_pandas(atom_ids)
H.sample_space.name = "Omega_generated"

print(H, "\n")
print(f"Generated sample space: {H.sample_space}")

Sigma algebra 'H':
        atom ID
sample         
a             0
b             0
c             0
d             1
e             2 

Generated sample space: Sample space 'Omega_generated':
['a', 'b', 'c', 'd', 'e']


### The trivial and power-set $\sigma$-algebras

On a sample space $\Omega$, there are two special $\sigma$-algebras that sit at the opposite ends of the $\sigma$-algebra lattice. The *power-set $\sigma$-algebra* on $\Omega$ consists of all subsets of $\Omega$. Its atoms are the singleton sets $\{\omega\}$ for each $\omega \in \Omega$. It is the finest $\sigma$-algebra on $\Omega$. The *trivial $\sigma$-algebra* on a set $\Omega$ consists of only the sets $\Omega$ and $\emptyset$. Its single atom is $\Omega$ itself. It is the coarsest $\sigma$-algebra on $\Omega$. Both may be created using dedicated class methods:

In [5]:
Omega = SampleSpace().from_sequence(size=4)  # Omega = {0, 1, 2, 3}
power_set = SigmaAlgebra.power_set(Omega)
trivial = SigmaAlgebra.trivial(Omega)

print(power_set, "\n")
print(trivial)

Sigma algebra 'power_set':
        atom ID
sample         
0             0
1             1
2             2
3             3 

Sigma algebra 'trivial':
        atom ID
sample         
0             0
1             0
2             0
3             0


Under the hood, the `power_set` method uses the sample points themselves as atom identifiers, while the `trivial` method uses the single atom identifier `0`.

### $\sigma$-algebra generated by an event

Given a nonempty event $A \subset \Omega$ which is not equal to $\Omega$ itself, the *$\sigma$-algebra generated by $A$* is the smallest $\sigma$-algebra $\sigma(A)$ on $\Omega$ that contains $A$. It has atoms $A$ and $A^c$ (provided both are nonempty). These special $\sigma$-algebras may be created using the `from_event` class method. For example, if we return to the $\sigma$-algebra $\mathcal{F}$ from above (printed below), we take $A = \{0, 1, 2\}$ and compute $\sigma(A)$ as follows:

In [6]:
A = F.get_event([0, 1, 2], name="A")
sigma_A = SigmaAlgebra.from_event(event=A)


print(F, "\n")
print(sigma_A)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             0
3             1
4             2 

Sigma algebra 'sigma(A)':
        atom ID
sample         
0             1
1             1
2             1
3             0
4             0


### $\sigma$-algebra generated by a random vector

Given a random vector $X: \Omega \to \mathbb{R}^d$, the *$\sigma$-algebra generated by $X$* is the smallest $\sigma$-algebra $\sigma(X)$ on $\Omega$ with respect to which $X$ is measurable. Its atoms are the nonempty level sets of $X$, i.e., the sets of the form $\{\omega \in \Omega : X(\omega) = x\}$ for $x\in \mathbb{R}^d$. These special $\sigma$-algebras may be created using the `from_random_vector` class method. For example:

In [7]:
from sigalg.core import RandomVector

X = RandomVector(domain=Omega, name="X").from_dict(  # Omega = {0, 1, 2, 3}
    {
        0: (1, 2),
        1: (1, 2),
        2: (3, 4),
        3: (5, 6),
    }
)
sigma_X = SigmaAlgebra.from_random_vector(rv=X)

print(sigma_X)

Sigma algebra 'sigma(X)':
       atom ID
sample        
0       (1, 2)
1       (1, 2)
2       (3, 4)
3       (5, 6)


Under the hood, SigAlg uses the values of the random vector itself as atom identifiers.

### Properties and attributes of $\sigma$-algebras

#### Sample spaces

Every instance of `SigmaAlgebra` carries its sample space as the `sample_space` attribute. Continuing with our $\sigma$-algebra $\mathcal{F}$ from above, we have:

In [8]:
print(F.sample_space)

Sample space 'Omega':
[0, 1, 2, 3, 4]


#### Dictionaries and data

No matter how they were created, all `SigmaAlgebra` instances also carry a `sample_id_to_atom_id` dictionary, mapping sample points to the corresponding atom identifiers.

In [9]:
print(F.sample_id_to_atom_id)  # created using from_dict
print(H.sample_id_to_atom_id)  # created using from_pandas

{0: 0, 1: 0, 2: 0, 3: 1, 4: 2}
{'a': 0, 'b': 0, 'c': 0, 'd': 1, 'e': 2}


Likewise, all instances also carry a `data` attribute, which is a `pd.Series` object mapping sample points to atom identifiers.

In [10]:
print(F.data, "\n")  # created using from_dict
print(H.data)  # created using from_pandas

sample
0    0
1    0
2    0
3    1
4    2
Name: atom ID, dtype: int64 

sample
a    0
b    0
c    0
d    1
e    2
Name: atom ID, dtype: int64


#### Atoms and atom identifiers

Access basic information about atoms:

In [11]:
print(F, "\n")
print(f"Number of atoms: {F.num_atoms}\n")
print(f"Atom IDs: {F.atom_ids}\n")
print(f"Atom ID to cardinality:\n{F.atom_id_to_cardinality}")

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             0
3             1
4             2 

Number of atoms: 3

Atom IDs: [0, 1, 2]

Atom ID to cardinality:
{0: 3, 1: 1, 2: 1}


The atom identifier mapping $\Omega \to I$ may be inverted, creating a mapping that sends an atom identifier to the list of sample points contained in the corresponding atom. This mapping is stored as the `atom_id_to_sample_ids` attribute of `SigmaAlgebra` instances. For example:

In [12]:
print(f"Atom ID to sample IDs:\n{F.atom_id_to_sample_ids}")

Atom ID to sample IDs:
{0: [0, 1, 2], 1: [3], 2: [4]}


Related to this last attribute, is a second attribute containing a dictionary mapping each atom identifier to the corresponding atom as an instance of `Event`.

In [13]:
for atom_id, event in F.atom_id_to_event.items():
    print(f"Atom {atom_id}:\n{event}\n")

Atom 0:
Event '0':
[0, 1, 2]

Atom 1:
Event '1':
[3]

Atom 2:
Event '2':
[4]



Notice that the names of the events are the atom identifiers.

If one simply wants a list of atoms as `Event` objects, one can use the `to_atoms` property:

In [14]:
for atom in F.to_atoms:
    print(atom)

Event '0':
[0, 1, 2]
Event '1':
[3]
Event '2':
[4]


### Methods

#### Getting the atom containing a sample point

For a given sample point, find which atom contains it:

In [15]:
print(F, "\n")
atom = F.get_atom_containing(sample_id=1)

print(atom)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             0
3             1
4             2 

Event 'A':
[0, 1, 2]


#### Checking if an event is measurable

An event $A$ is *$\mathcal{F}$-measurable* if it is a union of atoms of $\mathcal{F}$. Check measurability:

In [16]:
power_set = SigmaAlgebra.power_set(F.sample_space)
A = power_set.get_event([0, 1, 2], name="A")  # A is a measurable `Event` instance
B = power_set.get_event([2, 3], name="B")  # B is a non-measurable `Event` instance
C = [0, 2, 4]  # C is a non-measurable list of sample IDs (not an `Event` instance)

print(F, "\n")
print(f"Is A F-measurable? {F.is_measurable(event=A)}")
print(f"Is B F-measurable? {F.is_measurable(event=B)}")
print(f"Is C F-measurable? {F.is_measurable(event_list=C)}")

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             0
3             1
4             2 

Is A F-measurable? True
Is B F-measurable? False
Is C F-measurable? False


Equivalently, we may check measurability of an `Event` instance using the `in` operator, which is supposed to mimick the mathematical notation $A \in \mathcal{F}$:

In [17]:
print(f"Is A in F? {A in F}")
print(f"Is B in F? {B in F}")

Is A in F? True
Is B in F? False


### Operations and comparisons between $\sigma$-algebras

#### Join of $\sigma$-algebras

Given two $\sigma$-algebras $\mathcal{F}$ and $\mathcal{G}$, their *join* $\mathcal{F} \vee \mathcal{G}$ is the smallest $\sigma$-algebra that contains both $\mathcal{F}$ and $\mathcal{G}$. Its atoms are the nonempty intersections of atoms of $\mathcal{F}$ with atoms of $\mathcal{G}$. Thus, the atom identifiers of the join may be taken as ordered pairs of atom identifiers of $\mathcal{F}$ and $\mathcal{G}$, with the understanding that the pair $(i,j)$ corresponds to the intersection of the $i$-th atom of $\mathcal{F}$ with the $j$-th atom of $\mathcal{G}$.

In [18]:
Omega = SampleSpace().from_sequence(size=4)  # Omega = {0, 1, 2, 3}

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
    }
)

G = SigmaAlgebra(sample_space=Omega, name="G").from_dict(
    {
        0: 0,
        1: 1,
        2: 0,
        3: 1,
    }
)

print(F, "\n")
print(G, "\n")
print(F | G)

Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             1
3             1 

Sigma algebra 'G':
        atom ID
sample         
0             0
1             1
2             0
3             1 

Sigma algebra 'join':
       atom ID
sample        
0       (0, 0)
1       (0, 1)
2       (1, 0)
3       (1, 1)


#### Sub-$\sigma$-algebras

A $\sigma$-algebra $\mathcal{F}$ is a *sub-$\sigma$-algebra* of another $\sigma$-algebra $\mathcal{G}$ if $\mathcal{F} \subset \mathcal{G}$ as sets. Equivalently, every atom of $\mathcal{F}$ is a union of atoms of $\mathcal{G}$. In this case, we also say that $\mathcal{F}$ is *coarser* than $\mathcal{G}$, and that $\mathcal{G}$ is *finer* than $\mathcal{F}$.

In [19]:
Omega = SampleSpace().from_sequence(size=4)

trivial = SigmaAlgebra.trivial(sample_space=Omega, name="trivial")
middle = SigmaAlgebra(sample_space=Omega, name="middle").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
    }
)
power_set = SigmaAlgebra.power_set(sample_space=Omega, name="power_set")

print(trivial, "\n")
print(middle, "\n")
print(power_set, "\n")

print(f"Is trivial ⊆ middle? {trivial <= middle}")
print(f"Is trivial < middle? {trivial < middle}")
print(f"Is middle ⊆ power_set? {middle <= power_set}")
print(f"Is middle < power_set? {middle < power_set}")
print(f"Is trivial ⊆ power_set? {trivial <= power_set}")
print(f"Is power_set ⊆ trivial? {power_set <= trivial}")
print(f"Is middle ⊆ trivial? {middle <= trivial}")

Sigma algebra 'trivial':
        atom ID
sample         
0             0
1             0
2             0
3             0 

Sigma algebra 'middle':
        atom ID
sample         
0             0
1             0
2             1
3             1 

Sigma algebra 'power_set':
        atom ID
sample         
0             0
1             1
2             2
3             3 

Is trivial ⊆ middle? True
Is trivial < middle? True
Is middle ⊆ power_set? True
Is middle < power_set? True
Is trivial ⊆ power_set? True
Is power_set ⊆ trivial? False
Is middle ⊆ trivial? False


#### Equality

Two $\sigma$-algebras are equal if they are equal as sets. Equivalently, two $\sigma$-algebras are equal if they have the same atoms.

In [20]:
F1 = SigmaAlgebra(name="F1").from_dict(
    {
        0: 0,
        1: 0,
        2: 1,
    }
)

F2 = SigmaAlgebra(name="F2").from_dict(
    {
        0: 5,
        1: 5,
        2: 7,
    }
)

print(f"F1 == F2? {F1 == F2}")

F1 == F2? True


Note that the atom IDs themselves don't matter, only the partition structure.